In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [21]:
class CFG:
    TARGET = ['GGFM_f']
    N_FOLDS = 5
    RANDOM_STATE = 3
    TEST_SIZE = 0.2
    VAL_SIZE = 0.2
    
    COLORADO_PATH_TRAIN = './Data/Synt/Colorado_600_merge.csv'
    COLORADO_PATH_TEST = './Data/Synt/Colorado_60000_merge.csv'
    NSO_PATH_TRAIN = './Data/Synt/NSO_600_merge.csv'
    NSO_PATH_TEST = './Data/Synt/NSO_60000_merge.csv'
    
    SCALE_TARGET = True
    TARGET_SCALER = 'standard'
    
    # Параметры нейронной сети
    HIDDEN_LAYERS = [512, 512, 256, 256, 128, 64]
    DROPOUT_RATE = 0.05
    LEARNING_RATE = 1e-3
    EPOCHS = 2000
    PATIENCE = 100
    WEIGHT_DECAY = 1e-5
    BATCH_SIZE = 32
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [22]:
class PhysicsInformedNN(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim=1, dropout_rate=0.1):
        super().__init__()
        self.input_norm = nn.BatchNorm1d(input_dim)

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(ResidualBlock(prev_dim, hidden_dim, dropout_rate))
            prev_dim = hidden_dim

        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(prev_dim, output_dim)

    def forward(self, x):
        x = self.input_norm(x)
        x = self.network(x)
        return self.output_layer(x)


class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout_rate=0.1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim)
        self.bn1 = nn.BatchNorm1d(out_dim)
        self.fc2 = nn.Linear(out_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

        # Проекция для skip connection, если размерности не совпадают
        self.shortcut = nn.Identity()
        if in_dim != out_dim:
            self.shortcut = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.relu(self.bn1(self.fc1(x)))
        out = self.bn2(self.fc2(out))
        out = self.dropout(out)
        return self.relu(out + residual)

In [23]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

class DataLoader:
    def __init__(self, colorado: pd.DataFrame, nso: pd.DataFrame):
        self.colorado = colorado.copy()
        self.nso = nso
        self.log_features = []  # Список признаков для трансформации
        self.X = None
        self.y = None
        self.scaler = StandardScaler()  # Для нормализации признаков
        self.feature_names_ = []       # Для отслеживания имён признаков
        self.target_scaler = None
        self.feature_scaler = StandardScaler()

    @staticmethod
    def reduce_mem_usage(dataframe):
        """ 
        Уменьшает использование памяти dataframe путем преобразования типов данных
        с автоматическим пропуском временных столбцов
        """
        start_mem = dataframe.memory_usage().sum() / 1024**2
        print(f"Изначальное использование памяти: {start_mem:.2f} MB")
        
        for col in dataframe.columns:
            col_type = dataframe[col].dtype
            
            # Пропускаем временные столбцы и категориальные данные
            if str(col_type).startswith('datetime') or str(col_type) == 'category':
                continue
                
            if col_type != object:
                c_min = dataframe[col].min()
                c_max = dataframe[col].max()
                
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        dataframe[col] = dataframe[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        dataframe[col] = dataframe[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        dataframe[col] = dataframe[col].astype(np.int32)
                    elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        dataframe[col] = dataframe[col].astype(np.int64)
                else:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        dataframe[col] = dataframe[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        dataframe[col] = dataframe[col].astype(np.float32)
                    else:
                        dataframe[col] = dataframe[col].astype(np.float64)
            else:
                # Оптимизация строковых столбцов
                dataframe[col] = dataframe[col].astype('category')
        
        end_mem = dataframe.memory_usage().sum() / 1024**2
        print(f"Итоговое использование памяти: {end_mem:.2f} MB")
        print(f"Экономия {(start_mem - end_mem) / start_mem * 100:.1f}%")
        
        return dataframe

    def add_spherical_coordinates(self):
        """Добавляет сферические координаты (x, y, z) как направляющие косинусы"""
        lat_rad = np.radians(self.colorado['Latitude (deg)'])
        lon_rad = np.radians(self.colorado['Longitude (deg)'])
        self.colorado['sph_x'] = np.cos(lat_rad) * np.cos(lon_rad)
        self.colorado['sph_y'] = np.cos(lat_rad) * np.sin(lon_rad)
        self.colorado['sph_z'] = np.sin(lat_rad)

    def add_cartesian_coordinates(self):
        """Добавляет декартовы координаты в метрах (с учётом радиуса и высоты)"""
        R = 6378137  # средний радиус Земли в метрах
        lat_rad = np.radians(self.colorado['Latitude (deg)'])
        lon_rad = np.radians(self.colorado['Longitude (deg)'])
        h = self.colorado['Height (m)']
        
        self.colorado['cart_x'] = (R + h) * np.cos(lat_rad) * np.cos(lon_rad)
        self.colorado['cart_y'] = (R + h) * np.cos(lat_rad) * np.sin(lon_rad)
        self.colorado['cart_z'] = (R + h) * np.sin(lat_rad)

    def add_polynomial_features(self):
        """Добавляет полиномиальные признаки"""
        self.colorado['lat2'] = self.colorado['Latitude (deg)'] ** 2
        self.colorado['lon2'] = self.colorado['Longitude (deg)'] ** 2
        self.colorado['lat_lon'] = self.colorado['Latitude (deg)'] * self.colorado['Longitude (deg)']
        self.colorado['height2'] = self.colorado['Height (m)'] ** 2


    def add_distance_from_center(self):
        """Добавляет расстояние до центра облака точек (полезно для гравитации)"""
        center_lat = self.colorado['Latitude (deg)'].mean()
        center_lon = self.colorado['Longitude (deg)'].mean()
        self.colorado['dist_from_center'] = np.sqrt(
            (self.colorado['Latitude (deg)'] - center_lat)**2 +
            (self.colorado['Longitude (deg)'] - center_lon)**2
        )

    def scale_target(self, y, inverse=False):
        """Масштабирование целевой переменной"""
        if CFG.SCALE_TARGET:
            if self.target_scaler is None and not inverse:
                if CFG.TARGET_SCALER == 'standard':
                    self.target_scaler = StandardScaler()
                    return self.target_scaler.fit_transform(y.reshape(-1, 1)).flatten()
                elif CFG.TARGET_SCALER == 'minmax':
                    self.target_scaler = MinMaxScaler()
                    return self.target_scaler.fit_transform(y.reshape(-1, 1)).flatten()
                elif CFG.TARGET_SCALER == 'log':
                    # Логарифмическое преобразование + сдвиг если есть отрицательные значения
                    min_val = y.min()
                    if min_val <= 0:
                        shift = abs(min_val) + 1
                        self.target_shift = shift
                        return np.log(y + shift)
                    else:
                        return np.log(y)
            elif inverse and self.target_scaler is not None:
                if CFG.TARGET_SCALER == 'standard':
                    return self.target_scaler.inverse_transform(y.reshape(-1, 1)).flatten()
                elif CFG.TARGET_SCALER == 'minmax':
                    return self.target_scaler.inverse_transform(y.reshape(-1, 1)).flatten()
                elif CFG.TARGET_SCALER == 'log':
                    if hasattr(self, 'target_shift'):
                        return np.exp(y) - self.target_shift
                    else:
                        return np.exp(y)
        return y

    def load(self, option='colorado'):
        """Загрузка и обогащение данных"""
        print(f'Loading data')
        print(f'Choosed option: {option}')

        if option == 'compare':
            self.colorado = self.colorado.merge(self.nso, left_on='latitude(deg)', right_on='lat(deg)', how='outer')
        elif option == 'colorado':
            pass
        else:
            raise ValueError("Option must be 'colorado' or 'compare'")

        # Шаг 1: Добавление признаков
        self.add_spherical_coordinates()
        self.add_cartesian_coordinates()
        self.add_polynomial_features()
        self.add_distance_from_center()

        # Шаг 2: Уменьшение потребления памяти
        self.colorado = self.reduce_mem_usage(self.colorado)

        # Шаг 3: Подготовка X и y с масштабированием
        self.y = self.colorado[CFG.TARGET].values.flatten()
        
        # Масштабирование целевой переменной
        self.y = self.scale_target(self.y)
        
        # Выбор признаков и масштабирование
        feature_columns = [
            col for col in self.colorado.columns 
            if col not in ['Latitude (deg)', 'Longitude (deg)', 'Height (m)', CFG.TARGET]
        ]
        self.X = self.colorado[feature_columns].values
        self.feature_names_ = feature_columns
        
        # Масштабирование признаков
        self.X = self.feature_scaler.fit_transform(self.X)
        
        print(f"Целевая переменная после масштабирования: min={self.y.min():.2f}, max={self.y.max():.2f}")

        print(f"Количество признаков: {self.X.shape[1]}")
        print(f"Форма X: {self.X.shape}, Форма y: {self.y.shape}")
        print(f"Признаки: {self.feature_names_}")

In [24]:
def normal_gravitational_potential_grs80(latitude_deg, height):
    """
    Вычисляет нормальный гравитационный потенциал V (в м²/с²)
    по широте и высоте над эллипсоидом GRS80,
    используя формулу:
        V = GM/r * [1 - J2*(a/r)^2 * 0.5*(3*sin²φ - 1)]
    Расстояние r вычисляется аналитически через геодезические параметры.

    Параметры:
        latitude_deg (float): геодезическая широта в градусах
        height (float): высота над эллипсоидом в метрах

    Возвращает:
        float: нормальный гравитационный потенциал V [м²/с²]
    """
    # Константы GRS80
    a = 6378137.0              # большая полуось, м
    GM = 3.986004418e14        # гравитационная постоянная, м³/с²
    J2 = 1.08263e-3            # коэффициент J2
    e2 = 0.00669438002290       # квадрат первого эксцентриситета

    # Переводим широту в радианы
    phi = np.radians(latitude_deg)
    sin_phi = np.sin(phi)
    cos_phi = np.cos(phi)
    sin2_phi = sin_phi * sin_phi

    # Радиус кривизны в первом вертикале
    N = a / np.sqrt(1 - e2 * sin2_phi)

    # Вычисляем r — расстояние от центра Земли до точки
    # через компоненты (аналитически, без векторов)
    term_eq = (N + height) * cos_phi
    term_pol = (N * (1 - e2) + height) * sin_phi
    r = np.sqrt(term_eq**2 + term_pol**2)

    # Вычисляем J2-член: (a/r)^2 * 0.5*(3*sin²φ - 1)
    a_over_r = a / r
    P2_term = 0.5 * (3 * sin2_phi - 1)  # полином Лежандра 2-го порядка
    j2_correction = J2 * (a_over_r ** 2) * P2_term

    # Гравитационный потенциал
    V = (GM / r) * (1 - j2_correction)

    return V

In [25]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def calculate_laplacian_numeric(y_pred, coords, k=10, method='finite_difference_spherical'):
    """
    Вычисляет лапласиан ∇²Φ численно в каждой точке.
    
    Parameters:
    -----------
    y_pred : array (N,) — предсказанные значения потенциала Φ
    coords : array (N, 3) — координаты [lat (deg), lon (deg), height (m)]
    k : int — количество ближайших соседей для оценки
    method : str — метод:
        - 'finite_difference_cartesian' — для близких точек в 3D
        - 'finite_difference_spherical' — с учётом сферической геометрии
        - 'inverse_distance' — простая оценка через среднее (твой текущий метод)

    Returns:
    --------
    laplacian : array (N,) — оценка ∇²Φ в каждой точке
    """
    N = len(y_pred)
    laplacian = np.zeros(N)
    
    # Преобразуем координаты
    lat = np.radians(coords[:, 0])  # в радианы
    lon = np.radians(coords[:, 1])
    h = coords[:, 2]  # высота в метрах
    
    # Переводим в декартовы координаты (в метрах)
    R = 6378137 + h  # радиус от центра Земли
    x = R * np.cos(lat) * np.cos(lon)
    y = R * np.cos(lat) * np.sin(lat)  # исправлено: cos(lat) для y
    z = R * np.sin(lat)
    xyz = np.stack([x, y, z], axis=1)

    nbrs = NearestNeighbors(n_neighbors=k+1).fit(xyz)  # +1 включая саму точку
    distances, indices = nbrs.kneighbors(xyz)

    for i in range(N):
        center_idx = i
        neighbor_indices = indices[i]  # включает саму точку
        neighbor_dists = distances[i]
        
        # Исключаем саму точку (расстояние 0)
        mask = neighbor_dists > 1e-5
        if mask.sum() < 6:
            laplacian[i] = 0
            continue

        j = neighbor_indices[mask]
        dx = xyz[j] - xyz[i]  # векторы к соседям (в метрах)
        df = y_pred[j] - y_pred[i]  # разница в потенциале

        if method == 'finite_difference_cartesian':
            # Оценка лапласиана через среднее отклонение
            # ∇²Φ ≈ (1/N) Σ (Φ_j - Φ_i) / (d_j²/6)  → из разложения Тейлора
            d_sq = np.sum(dx**2, axis=1) + 1e-8
            weights = 1 / d_sq
            weights /= weights.sum()
            laplacian[i] = 6 * np.dot(weights, df) / np.mean(d_sq)  # грубая оценка

        elif method == 'finite_difference_spherical':
            # Более точная оценка через локальную сферическую систему
            # Аппроксимируем как ∇²Φ ≈ (Σ w_j (Φ_j - Φ_i)) / σ²
            d = np.linalg.norm(dx, axis=1)
            sigma = np.mean(d)
            laplacian[i] = np.sum(df) / (sigma**2 + 1e-8)

        elif method == 'inverse_distance':
            # Твой текущий метод: ∇²Φ ≈ Φ_i - <Φ_соседей>
            weights = 1 / (neighbor_dists[mask] + 1e-8)
            weights /= weights.sum()
            avg_neighbor = np.dot(y_pred[j], weights)
            laplacian[i] = y_pred[i] - avg_neighbor  # может быть смещённым

    return laplacian

In [26]:
# Класс для обучения с физическими ограничениями
class PhysicsInformedTrainer:
    def __init__(self, model, device, alpha=1.0):
        self.model = model.to(device)
        self.device = device
        self.alpha = alpha
        
    def combined_loss(self, y_true, y_pred, coords, k=10):
        mse_loss = nn.MSELoss()(y_pred, y_true)

        # Если мало точек — не считаем лапласиан
        if len(y_pred) < 2:
            laplace_loss = torch.tensor(0.0, device=self.device)
        else:
            k_safe = min(k, len(y_pred) - 1)
            if k_safe < 1:
                laplace_loss = torch.tensor(0.0, device=self.device)
            else:
                laplacian = calculate_laplacian_numeric(
                    y_pred.detach().cpu().numpy().flatten(),
                    coords.cpu().numpy() if torch.is_tensor(coords) else coords,
                    k=k_safe
                )
                laplace_loss = torch.tensor(np.mean(laplacian ** 2), device=self.device)

        total_loss = mse_loss + self.alpha * laplace_loss
        return total_loss, mse_loss, laplace_loss
    
    def train_epoch(self, train_loader, optimizer, coords_train, k=10):
        self.model.train()
        total_loss = 0
        total_mse = 0
        total_laplace = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(self.device), target.to(self.device)
            
            optimizer.zero_grad()
            output = self.model(data)
            
            # Получаем соответствующие координаты для батча
            batch_indices = batch_idx * train_loader.batch_size
            batch_coords = coords_train[batch_indices:batch_indices + len(data)]
            batch_coords = torch.tensor(batch_coords, device=self.device)
            
            loss, mse_loss, laplace_loss = self.combined_loss(
                target, output, batch_coords, k=k
            )
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            total_mse += mse_loss.item()
            total_laplace += laplace_loss.item()
        
        return total_loss / len(train_loader), total_mse / len(train_loader), total_laplace / len(train_loader)
    
    def validate(self, val_loader, coords_val, k=10):
        self.model.eval()
        total_loss = 0
        total_mse = 0
        total_laplace = 0
        
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(val_loader):
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                
                batch_indices = batch_idx * val_loader.batch_size
                batch_coords = coords_val[batch_indices:batch_indices + len(data)]
                batch_coords = torch.tensor(batch_coords, device=self.device)
                
                loss, mse_loss, laplace_loss = self.combined_loss(
                    target, output, batch_coords, k=k
                )
                
                total_loss += loss.item()
                total_mse += mse_loss.item()
                total_laplace += laplace_loss.item()
        
        return total_loss / len(val_loader), total_mse / len(val_loader), total_laplace / len(val_loader)


In [27]:
# Функция для кросс-валидации
def cross_validate_with_physics(X, y, coords, model_params, n_folds=5, alpha=1.0):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=CFG.RANDOM_STATE)
    
    results = {
        'total_loss': [], 'mse_loss': [], 'laplace_loss': [], 'r2': [], 'models': []
    }
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        print(f"Fold {fold + 1}/{n_folds}")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        coords_train, coords_val = coords[train_idx], coords[val_idx]
        
        # Преобразуем в тензоры
        X_train_tensor = torch.FloatTensor(X_train.values)
        y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
        X_val_tensor = torch.FloatTensor(X_val.values)
        y_val_tensor = torch.FloatTensor(y_val.values).reshape(-1, 1)
        
        # Создаем DataLoader
        train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True
        )
        
        val_dataset = torch.utils.data.TensorDataset(X_val_tensor, y_val_tensor)
        val_loader = torch.utils.data.DataLoader(
            val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False
        )
        
        # Создаем и обучаем модель
        model = PhysicsInformedNN(
            input_dim=X.shape[1],
            hidden_layers=CFG.HIDDEN_LAYERS,
            output_dim=1,
            dropout_rate=CFG.DROPOUT_RATE
        )
        
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CFG.LEARNING_RATE,
            weight_decay=CFG.WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2, eta_min=1e-7
        )
        
        trainer = PhysicsInformedTrainer(model, CFG.DEVICE, alpha=alpha)
        
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(CFG.EPOCHS):
            train_loss, train_mse, train_laplace = trainer.train_epoch(
                train_loader, optimizer, coords_train
            )
            
            val_loss, val_mse, val_laplace = trainer.validate(
                val_loader, coords_val
            )
            
            scheduler.step(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
            
            if patience_counter >= CFG.PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break
        
        # Загружаем лучшую модель
        model.load_state_dict(best_model_state)
        
        # Оценка на валидации
        model.eval()
        with torch.no_grad():
            y_pred = model(X_val_tensor.to(CFG.DEVICE)).cpu().numpy()
        
        r2 = r2_score(y_val, y_pred)
        
        results['total_loss'].append(best_val_loss)
        results['mse_loss'].append(val_mse)
        results['laplace_loss'].append(val_laplace)
        results['r2'].append(r2)
        results['models'].append(model)
        
        print(f"Fold {fold + 1} - MSE: {val_mse:.4f}, "
              f"Laplace: {val_laplace:.4f}, R²: {r2:.4f}")
    
    # Усреднение результатов
    avg_results = {metric: np.mean(values) for metric, values in results.items() 
                  if metric != 'models'}
    
    print(f"\nAverage CV Results:")
    print(f"MSE: {avg_results['mse_loss']:.4f} ± {np.std(results['mse_loss']):.4f}")
    print(f"Laplace Loss: {avg_results['laplace_loss']:.4f} ± {np.std(results['laplace_loss']):.4f}")
    print(f"Total Loss: {avg_results['total_loss']:.4f} ± {np.std(results['total_loss']):.4f}")
    print(f"R²: {avg_results['r2']:.4f} ± {np.std(results['r2']):.4f}")
    
    return results, avg_results


In [28]:
loader = DataLoader(
    colorado=pd.read_csv(CFG.COLORADO_PATH_TRAIN, skipinitialspace=True, index_col='index'),
    nso=pd.read_csv(CFG.NSO_PATH_TRAIN, skipinitialspace=True, index_col='index'),
)
loader.load()

normal_potentials = normal_gravitational_potential_grs80(
    loader.colorado["Latitude (deg)"].values, 
    loader.colorado["Height (m)"].values
)

loader.colorado[CFG.TARGET] = loader.colorado[CFG.TARGET].values - pd.DataFrame(normal_potentials)
loader.colorado.drop(columns="GGFM_g", inplace=True)

features = pd.DataFrame(loader.X, columns=loader.feature_names_)
target = pd.Series(loader.y)
coords = loader.colorado[['Latitude (deg)', 'Longitude (deg)', 'Height (m)']].values

# Разделение на train/test
X_train, X_test, y_train, y_test, coords_train, coords_test = train_test_split(
    features, target, coords, 
    test_size=CFG.TEST_SIZE, 
    random_state=CFG.RANDOM_STATE
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# Кросс-валидация с физическими ограничениями
cv_results, avg_cv_results = cross_validate_with_physics(
    X_train, y_train, coords_train, 
    model_params={}, n_folds=CFG.N_FOLDS, alpha=1.0
)

# Выбор лучшей модели
best_model_idx = np.argmin(cv_results['total_loss'])
best_model = cv_results['models'][best_model_idx]

# Финальная оценка на тестовом наборе
print("=== FINAL TEST EVALUATION ===")

X_test_tensor = torch.FloatTensor(X_test.values).to(CFG.DEVICE)
y_test_tensor = torch.FloatTensor(y_test.values.reshape(-1, 1)).to(CFG.DEVICE)

best_model.eval()
with torch.no_grad():
    y_pred = best_model(X_test_tensor).cpu().numpy()

test_mse = mean_squared_error(y_test, y_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_pred)

# Вычисляем лапласиан для тестовых данных
test_laplacian = calculate_laplacian_numeric(y_pred.flatten(), coords_test)
test_laplace_loss = np.mean(test_laplacian ** 2)

print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test Laplace Loss: {test_laplace_loss:.4f}")
print(f"Test R²: {test_r2:.4f}")

# Сохранение модели
torch.save(best_model.state_dict(), './models/physics_informed_nn.pth')
print("Модель сохранена: ./models/physics_informed_nn.pth")

Loading data
Choosed option: colorado
Изначальное использование памяти: 0.84 MB
Итоговое использование памяти: 0.31 MB
Экономия 63.5%
Целевая переменная после масштабирования: min=-2.01, max=2.46
Количество признаков: 20
Форма X: (4608, 20), Форма y: (4608,)
Признаки: ['Height relief (m)', 'Height geoid (m)', 'r (m)', 'GGFM_f', 'GGFM_g', 'TGFM_f', 'TGFM_g', 'Density (kg/m^3)', 'Density STD (kg/m^3)', 'sph_x', 'sph_y', 'sph_z', 'cart_x', 'cart_y', 'cart_z', 'lat2', 'lon2', 'lat_lon', 'height2', 'dist_from_center']
Train size: 3686, Test size: 922
Fold 1/5
Early stopping at epoch 130
Fold 1 - MSE: 0.0116, Laplace: 0.0000, R²: 0.9881
Fold 2/5
Early stopping at epoch 147
Fold 2 - MSE: 0.0058, Laplace: 0.0000, R²: 0.9943
Fold 3/5
Early stopping at epoch 277
Fold 3 - MSE: 0.0252, Laplace: 0.0000, R²: 0.9760
Fold 4/5
Early stopping at epoch 246
Fold 4 - MSE: 0.0129, Laplace: 0.0000, R²: 0.9864
Fold 5/5
Early stopping at epoch 282
Fold 5 - MSE: 0.0124, Laplace: 0.0000, R²: 0.9876

Average CV R